# CHIMERA Task 1: full baseline run (Kaggle + Ollama)

The notebook used for the baseline run on 18 Oct 2026 (91 labeled cases, results in `results/baseline.md`).
Cells 1–6 are from `KAGGLE_SETUP.md`, cells A–D from `RUN_ALL_CASES.md`. Explanations and troubleshooting are in those docs and in `baseline/README.md`.

**Session settings:** Accelerator GPU T4 x2, Internet on, Persistence none, the private dataset added as input, the `HF_TOKEN` secret attached.

**First time:** run interactively, and in cell B set `MAX_BATCHES = 1` for a ~15 min timing test. Then set it back to `None` and run cell B again: cases that already have output are skipped.

## Cell 1: check the GPU and find the data path

Copy the part of the `task1` path **before** `/task1` into `DATA_ROOT` in cell 2.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!find /kaggle/input -maxdepth 6 -type d -name task1

## Cell 2: secrets and paths

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

DATA_ROOT    = "/kaggle/input/datasets/vladyslavsh0/chimera-agent-challenge-task-1"   # from cell 1; Oleksii's path differs
LABELS       = f"{DATA_ROOT}/task1/labels"      # never passed to the agent
OUT_DIR      = "/kaggle/working/output"
REPO         = "/kaggle/temp/chimera-agent-baseline"   # /kaggle/temp is NOT saved as output; /kaggle/working is.
                                                       # Cloning into /kaggle/working put the 1.2 GB embedding model into every
                                                       # version's output, which made attaching that output as input very slow
OLLAMA_MODEL = "gemma4:e2b-it-q8_0"

## Cell 3: clone and install the baseline (~2–3 min, no vLLM)

Red `ERROR: pip's dependency resolver ...` lines about Kaggle's preinstalled packages can be ignored.

In [ ]:
![ -d {REPO} ] || git clone --depth 1 https://github.com/DIAGNijmegen/chimera-agent-baseline.git {REPO}
%cd {REPO}
!pip install -q -e .   # run this in EVERY new session: installed packages don't survive, even if the repo folder does
!pip install -q "transformers==5.12.1"   # pin from the baseline's requirements.txt

## Cell 4: download the guideline-search model (~1.2 GB)

In [ ]:
!python scripts/download_embedding_model.py
!ls resources/embedding_model | head -3

## Cell 5: install and start Ollama

Needs Ollama ≥ 0.22.0 for Gemma 4 tool calling. After a session restart, rerun cells 2, 3 and 5.

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama_env = {**os.environ,
              "OLLAMA_HOST": "127.0.0.1:11434",
              "OLLAMA_CONTEXT_LENGTH": "16384"}   # Ollama's default context is far too short for the agent
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                               stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

## Cell 6: download the language model (~6 GB, ~1–2 min)

In [ ]:
!ollama pull {OLLAMA_MODEL}
!ollama list

## Cell A: record what we ran (for the report)

In [ ]:
import json, subprocess, datetime
run_info = {
    "date": datetime.datetime.now().isoformat(timespec="minutes"),
    "baseline_commit": subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    "ollama_version": subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip(),
    "ollama_model": OLLAMA_MODEL,
    "ollama_model_id": subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout.strip(),
    "context_length": ollama_env["OLLAMA_CONTEXT_LENGTH"],
    "generation": "baseline defaults (temperature 1.0, top_p 0.95, max_new_tokens 4096)",
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
}
json.dump(run_info, open("/kaggle/working/run_info.json", "w"), indent=2)
print(json.dumps(run_info, indent=2))

## Cell B: the runner (resumable)

In [ ]:
import os, json, time, datetime, subprocess, urllib.request
from pathlib import Path

SCOPE = "labeled"   # "labeled" = 91 cases with reference answers, "all" = all 195
BATCH = 10
MAX_BATCHES = None  # set to 1 for a timing test, None = run everything

INPUT_DIR   = Path(DATA_ROOT) / "task1" / "agent_input"
OUT         = Path(OUT_DIR)
LOGS        = Path("/kaggle/working/logs"); LOGS.mkdir(parents=True, exist_ok=True)
FAILED_FILE = Path("/kaggle/working/failed_cases.json")

labeled = sorted(os.listdir(LABELS))
cases = labeled if SCOPE == "labeled" else labeled + [c for c in sorted(os.listdir(INPUT_DIR)) if c not in set(labeled)]

def done(c):
    d = OUT / "task1" / c
    return (d / "prostate-biopsy-decision.json").exists() and (d / "prostate-biopsy-decision-reasoning.json").exists()

def ollama_alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=5)
        return True
    except Exception:
        return False

def ensure_ollama():
    global ollama_proc
    if ollama_alive():
        return
    print("Ollama not responding, restarting it", flush=True)
    ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                                   stdout=open("/kaggle/working/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(30):
        time.sleep(2)
        if ollama_alive():
            return
    raise RuntimeError("Ollama did not start, see /kaggle/working/ollama.log")

import sys
# Run the baseline with the notebook's own Python, and find the package via PYTHONPATH even if
# `pip install -e .` didn't run in this session (installed packages don't survive a new session)
RUN_ENV = {**os.environ, "PYTHONPATH": f"{REPO}/src"}

def run_baseline(pids, log_path):
    cmd = [sys.executable, "-m", "chimera_agent_baseline.run",
           "agent.tasks=[1]", "agent.pids=[" + ",".join(pids) + "]",
           f"paths.data_root={DATA_ROOT}", f"paths.output_dir={OUT_DIR}",
           "paths.embedding_model_dir=resources/embedding_model",
           "model.provider=openai", "model.base_url=http://127.0.0.1:11434/v1",
           f"model.model_id={OLLAMA_MODEL}"]
    with open(log_path, "w") as f:
        return subprocess.run(cmd, cwd=REPO, env=RUN_ENV, stdout=f, stderr=subprocess.STDOUT).returncode

# Preflight: catch setup problems in seconds, before any batch runs
check = subprocess.run([sys.executable, "-c", "import chimera_agent_baseline.run, langgraph, chromadb"],
                       cwd=REPO, env=RUN_ENV, capture_output=True, text=True)
assert check.returncode == 0, f"Baseline can't be imported, run the install cell (cell 3) first:\n{check.stderr[-1500:]}"
assert (INPUT_DIR).is_dir(), f"No agent_input under DATA_ROOT: {INPUT_DIR}"
assert (Path(REPO) / "resources/embedding_model").is_dir(), "Embedding model missing, run cell 4"
ensure_ollama()
assert OLLAMA_MODEL in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout, \
    f"{OLLAMA_MODEL} not pulled, run cell 6"
print("Preflight OK")

failed = json.loads(FAILED_FILE.read_text()) if FAILED_FILE.exists() else {}
todo = [c for c in cases if not done(c) and c not in failed]
n_batches = -(-len(todo) // BATCH)
if MAX_BATCHES:
    n_batches = min(n_batches, MAX_BATCHES)
print(f"{len(cases)} cases in scope | {sum(map(done, cases))} done | {len(failed)} failed earlier | {len(todo)} to run in {n_batches} batches", flush=True)

def tail(path, n=25):
    return "".join(open(path).readlines()[-n:])

t_start, n_new = time.time(), 0
for b in range(n_batches):
    batch = todo[b * BATCH:(b + 1) * BATCH]
    ensure_ollama()
    t0 = time.time()
    batch_log = LOGS / f"batch_{b:03d}.log"
    run_baseline(batch, batch_log)
    # Circuit breaker: if a whole batch produced nothing, something is broken for every case
    # (paths, server, install). Stop instead of retrying every case and burning GPU hours.
    if not any(done(c) for c in batch):
        raise RuntimeError(f"Batch {b} produced no outputs at all, stopping. End of {batch_log}:\n\n{tail(batch_log)}")
    # The baseline stops at the first case that fails, so retry the missing ones one by one
    for c in [c for c in batch if not done(c)]:
        ensure_ollama()
        rc = run_baseline([c], LOGS / f"case_{c}.log")
        if not done(c):
            failed[c] = f"exit code {rc}, see logs/case_{c}.log"
            FAILED_FILE.write_text(json.dumps(failed, indent=2))
    n_new += sum(done(c) for c in batch)
    per_case = (time.time() - t_start) / max(n_new, 1)
    left = sum(not done(c) and c not in failed for c in cases)
    print(f"[{datetime.datetime.now():%H:%M}] batch {b + 1}/{n_batches} took {time.time() - t0:.0f}s | "
          f"{sum(map(done, cases))}/{len(cases)} done, {len(failed)} failed | "
          f"{per_case:.0f}s/case, ~{per_case * left / 60:.0f} min left", flush=True)

print("Finished. Failed cases:", list(failed) or "none")

## GPU check (cell 9 of KAGGLE_SETUP): check that Ollama used the GPU
Good: `offloaded N/N layers to GPU` and several GB of GPU memory in use. 0 layers means it ran on the CPU.

In [ ]:
!grep -iE "offload|layers.*gpu|CUDA|compute" /kaggle/working/ollama.log | tail -8
!nvidia-smi --query-gpu=memory.used --format=csv

## Cell C: metrics on the labeled cases

In [ ]:
import os, json, collections
from pathlib import Path
OUT, INPUT_DIR = Path(OUT_DIR), Path(DATA_ROOT) / "task1" / "agent_input"
def done(c):
    d = OUT / "task1" / c
    return (d / "prostate-biopsy-decision.json").exists() and (d / "prostate-biopsy-decision-reasoning.json").exists()

rows = []
for c in sorted(os.listdir(LABELS)):
    ref = json.load(open(f"{LABELS}/{c}/prostate-biopsy-decision.json"))
    bx = json.load(open(INPUT_DIR / c / "structured-prompt.json")).get("bx")
    p = OUT / "task1" / c
    pred = json.load(open(p / "prostate-biopsy-decision.json")) if done(c) else None
    reas = json.load(open(p / "prostate-biopsy-decision-reasoning.json")) if done(c) else None
    rows.append({"case": c, "ref": ref, "pred": pred, "bx": bx, "reas": reas})

def metrics(rs):
    # A case without output counts as wrong (as on the leaderboard); it can't be a false positive
    tp = sum(r["ref"] == "yes" and r["pred"] == "yes" for r in rs)
    fp = sum(r["ref"] == "no" and r["pred"] == "yes" for r in rs)
    fn = sum(r["ref"] == "yes" and r["pred"] != "yes" for r in rs)
    acc = sum(r["ref"] == r["pred"] for r in rs) / len(rs)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    return acc, prec, rec, f1

n_ok = sum(r["pred"] is not None for r in rows)
acc, prec, rec, f1 = metrics(rows)
always_yes = metrics([{**r, "pred": "yes"} for r in rows])
print(f"Coverage: {n_ok}/{len(rows)} cases have output")
print(f"Baseline     acc {acc:.3f} | precision {prec:.3f} | recall {rec:.3f} | F1(yes) {f1:.3f}")
print(f"Always 'yes' acc {always_yes[0]:.3f} | precision {always_yes[1]:.3f} | recall {always_yes[2]:.3f} | F1(yes) {always_yes[3]:.3f}")

print("\nConfusion (ref → pred):", dict(collections.Counter((r["ref"], r["pred"]) for r in rows)))
print("\nBy prior biopsy result (bx):")
for bx in sorted({r["bx"] for r in rows}, key=str):
    rs = [r for r in rows if r["bx"] == bx]
    a, _, _, f = metrics(rs)
    print(f"  bx={bx!s:9} n={len(rs):3} | acc {a:.3f} | F1(yes) {f:.3f}")

done_rows = [r for r in rows if r["reas"]]
print("\nConfidence:", dict(collections.Counter(r["reas"]["confidence"] for r in done_rows)))
print("Avg tools called per case:", round(sum(len(r["reas"]["reveal_sequence"]) for r in done_rows) / max(len(done_rows), 1), 2))
print("Tool usage:", dict(collections.Counter(t for r in done_rows for t in r["reas"]["reveal_sequence"])))

json.dump([{k: v for k, v in r.items() if k != "reas"} for r in rows],
          open("/kaggle/working/predictions_vs_reference.json", "w"), indent=2)

## Cell D: save everything

**Download `baseline_run.zip` right away** (Output → `/kaggle/working`). Unpack it locally into `results/baseline/` (gitignored, patient data).

In [ ]:
!cd /kaggle/working && zip -qr baseline_run.zip output logs run_info.json predictions_vs_reference.json failed_cases.json ollama.log 2>/dev/null; ls -lh baseline_run.zip